# Inworld Voice Isolator v1 — 48 kHz voice isolation on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Inworld-Voice-Isolator-48k.ipynb)

[`inworld/voice-isolator`](https://huggingface.co/inworld/voice-isolator) is Inworld's **Voice Isolator v1** (Apache-2.0, public): two 48 kHz checkpoints that share one architecture. `voice-isolator-v1` is **64.0M** parameters for recorded audio (offline: every frame sees its whole **8 s** chunk; long inputs render in 8 s chunks overlapping by **2 s**). `voice-isolator-v1-realtime` is **58.2M** for live audio (causal stream with a trained look-ahead of **0, 20 or 40 ms**; the card's algorithmic latency is **63 ms** at 20 ms of look-ahead). Hub files are `denoiser.safetensors` (**256.3 MB**) and `denoiser_realtime.safetensors` (**232.8 MB**). Input is one channel at **48 kHz**.

**Why this model.** The [card](https://huggingface.co/inworld/voice-isolator) is for cleaning recordings before playback or publication, curating speech datasets and voice pipelines where the speaker's identity must survive, and (with the realtime model) live calls and streams. Subtract the isolated voice from the input to extract background noise or a backing-track residual. Version 1 does **denoising** (noise suppression): it removes what surrounds the voice and restores nothing the recording lost, such as missing bandwidth. It is **not** a speaker selector. Restoration is planned for version 2.

See [Use cases](https://github.com/donvito/notebooks/blob/main/docs/Inworld-Voice-Isolator-48k.md) for when this notebook is a good fit and when it is not.

**Licence:** Apache-2.0 on code and weights. Third-party notices are in [`NOTICE`](https://huggingface.co/inworld/voice-isolator/blob/main/NOTICE). Public weights — no Hugging Face token.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. A free T4 has about **15 GB** of VRAM, **no native bfloat16** (`sm_75`), and **no FlashAttention-2**. The card: CUDA is used when available, else CPU. Inworld tested Ampere, Hopper and Blackwell GPUs (A100, H100, B200, B300) and CPU; **smaller GPUs are not tested yet**. This notebook still uses CUDA when a T4 is attached. Default `precision="auto"` is bfloat16 autocast on CUDA (the training precision) and float32 elsewhere; on T4 this notebook passes **`precision="fp16"`**. The STFT, iSTFT and selective scan stay in float32 in every case. Do **not** install `inworld-voice-isolation[fast]` (CUDA Mamba kernels need Ampere+). Do **not** install `flash-attn`. First run downloads the two checkpoints (~0.5 GB together). Peak VRAM is printed at the end.

**Not in this notebook:** the Gradio interactive demo, VIBE scoring (the eval set is not distributed), version 2 restoration, or natural-language prompting.

**Links:** [model card](https://huggingface.co/inworld/voice-isolator) · [GitHub](https://github.com/inworld-ai/voice-isolator) · [PyPI](https://pypi.org/project/inworld-voice-isolation/) · [official cleanup notebook](https://github.com/inworld-ai/voice-isolator/blob/main/examples/audio_cleanup.ipynb) · [blog](https://inworld.ai/blog/introducing-voice-isolator) · [announcement](https://x.com/inworld/status/2123120546908926300) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/Inworld-Voice-Isolator-48k.md)


## 1. Install (keep Colab's torch)

The [card](https://huggingface.co/inworld/voice-isolator#quick-start) install is:

```bash
pip install inworld-voice-isolation
```

Python 3.10+ and PyTorch 2.1+ (the wheel also needs `torchaudio`, `safetensors`, `soundfile`, `numpy`, `huggingface-hub`). Current Colab already has a CUDA `torch`. This cell installs the package and does **not** replace Colab's torch. Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled `torch`.

Do **not** install `inworld-voice-isolation[fast]` on a T4 — that extra pulls `causal-conv1d` and `mamba-ssm`, which need Ampere-or-newer GPUs. A Colab T4 is Turing (`sm_75`). Do **not** install `flash-attn`.


In [ ]:
%pip install -q inworld-voice-isolation

# Card: pip install inworld-voice-isolation
# Keep Colab's CUDA torch. Use %pip, not python -I.
# Do NOT install inworld-voice-isolation[fast] (T4 cannot run the CUDA Mamba extra).
# Do NOT install flash-attn.

import importlib.metadata as md
import sys

import torch

print("python", sys.version.split()[0])
print("torch (keep Colab's wheel)", torch.__version__, "| cuda", torch.version.cuda)
print("inworld-voice-isolation", md.version("inworld-voice-isolation"))
print("torchaudio", md.version("torchaudio"))
print("soundfile", md.version("soundfile"))
print("safetensors", md.version("safetensors"))


> Restart is usually unnecessary: this install does not pin `transformers` or replace torch. If a later cell cannot import `inworld.voice_isolation`, use *Runtime → Restart session* and continue from the next cell. You do not need to rerun the install unless it failed.


## 2. Check the GPU

Print `torch.cuda.is_available()`, the device name, and free VRAM. A T4 is the intended path. The card also runs on CPU, so this cell continues on CPU instead of stopping.


In [ ]:
import gc
import os
import shutil
import time
from pathlib import Path

import numpy as np
import torch
from IPython.display import Audio, display

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)

if torch.cuda.is_available():
    DEVICE = "cuda"
    major, minor = torch.cuda.get_device_capability()
    free_b, total_b = torch.cuda.mem_get_info()
    try:
        native_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
    except TypeError:
        native_bf16 = major >= 8
    # Card default precision="auto" is bf16 autocast on CUDA. T4 (sm_75) has
    # no native bfloat16, so this notebook uses the documented fp16 option.
    PRECISION = "fp16" if major < 8 else "auto"
    print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
    print(f"VRAM free / total: {free_b / 1024**3:.2f} / {total_b / 1024**3:.1f} GB")
    print("native bf16:", native_bf16, "| FlashAttention-2: off")
    print("precision:", PRECISION, "(STFT / iSTFT / selective scan stay float32)")
    if major < 8:
        print("T4 path: fp16 autocast on sm_75. Do not install the [fast] extra.")
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
else:
    DEVICE = "cpu"
    PRECISION = "auto"  # float32 off CUDA
    print("GPU name: none")
    print("VRAM free: n/a")
    print("No GPU — the card also runs on CPU. A free T4 is faster")
    print("(*Runtime > Change runtime type > T4 GPU*).")

print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")
print("device:", DEVICE, "| precision:", PRECISION)


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        peak = torch.cuda.max_memory_allocated() / 1024**3
        free_now, total_now = torch.cuda.mem_get_info()
        print(
            f"VRAM allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB | "
            f"peak: {peak:.2f} GB | free: {free_now / 1024**3:.2f}/{total_now / 1024**3:.1f} GB"
            f"{suffix}"
        )
    else:
        import resource

        rss_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        print(f"CPU RSS peak: {rss_kb / 1024 / 1024:.2f} GB{suffix}")


## 3. Load both checkpoints

Same call as the [card](https://huggingface.co/inworld/voice-isolator#quick-start): `VoiceIsolator("inworld/voice-isolator")` for the offline model, and `mode="streaming", lookahead_frames=2` for the realtime model (20 ms of look-ahead). First run downloads the pinned Hub files and verifies their checksums. No account or sign-in is needed.

`voice-isolate-doctor` reports Python, PyTorch, and the device inference will use. A `FAIL` line names what to fix.


In [ ]:
import subprocess
import sys

from inworld import voice_isolation

print("inworld.voice_isolation", voice_isolation.__version__)
print("repo:", "inworld/voice-isolator")

subprocess.run(
    [sys.executable, "-m", "inworld.voice_isolation.doctor"],
    check=False,
)

offline = voice_isolation.VoiceIsolator(
    "inworld/voice-isolator",
    device=DEVICE,
    precision=PRECISION,
)
print()
print("offline loaded")
print("  device:", offline.device, "| precision:", offline.precision)
print("  sample_rate:", offline.sample_rate, "Hz | mode:", offline.mode)
n_off = sum(p.numel() for p in offline.model.parameters())
print(f"  parameters: {n_off / 1e6:.1f}M  (card: 64.0M, denoiser.safetensors 256.3 MB)")
print_vram("after offline load")

live = voice_isolation.VoiceIsolator(
    "inworld/voice-isolator",
    device=DEVICE,
    precision=PRECISION,
    mode="streaming",
    lookahead_frames=2,
)
print()
print("realtime loaded")
print("  device:", live.device, "| precision:", live.precision)
print("  sample_rate:", live.sample_rate, "Hz | mode:", live.mode)
print("  lookahead_frames:", live.lookahead_frames, "| choices:", live.lookahead_choices)
print(f"  frame_seconds: {live.frame_seconds * 1000:g} ms")
print(f"  algorithmic latency: {live.latency_samples() / live.sample_rate * 1000:.0f} ms")
n_rt = sum(p.numel() for p in live.model.parameters())
print(f"  parameters: {n_rt / 1e6:.1f}M  (card: 58.2M, denoiser_realtime.safetensors 232.8 MB)")
print_vram("after both loads")


## 4. Demo 1 — offline isolation of the bundled sample

The card's noisy example is [`sample_14.wav`](https://raw.githubusercontent.com/inworld-ai/voice-isolator/main/demo/examples/sample_14.wav). `isolate` returns the voice at the input's sample rate, length and level. Subtracting it from the input gives what the model removed. Both outputs are float32 at 48 kHz; together they reconstruct the aligned mono input.

Players keep the true levels (no normalization). Listen for:

- **Noise.** Is the background gone, including in the pauses between words?
- **Speech.** Does the voice still sound natural? Breaths and other non-verbal sounds are kept on purpose.
- **Removed background.** It should hold the noise and little or no voice. Voice you hear here is voice the model took out (the demo README notes the background can contain leakage).


In [ ]:
SAMPLE_URL = (
    "https://raw.githubusercontent.com/inworld-ai/voice-isolator/"
    "main/demo/examples/sample_14.wav"
)
SAMPLE_PATH = Path("sample_14.wav")

if not SAMPLE_PATH.is_file():
    import urllib.request

    urllib.request.urlretrieve(SAMPLE_URL, SAMPLE_PATH)
    print("downloaded", SAMPLE_PATH, f"({SAMPLE_PATH.stat().st_size / 1024:.1f} KB)")
else:
    print("using cached", SAMPLE_PATH.resolve())

wav, sample_rate = voice_isolation.load_audio(str(SAMPLE_PATH))
print(
    "input:",
    tuple(wav.shape),
    f"dtype={wav.dtype}",
    f"sr={sample_rate} Hz",
    f"duration={len(wav) / sample_rate:.2f} s",
)


def to_playable(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().float().clamp(-1, 1)
        return np.asarray(x.numpy())
    return np.clip(np.asarray(x, dtype=np.float32), -1.0, 1.0)


def play(label, audio, rate):
    print(label)
    display(Audio(to_playable(audio), rate=rate, normalize=False))


if DEVICE == "cuda":
    torch.cuda.synchronize()
t0 = time.perf_counter()
voice = offline.isolate(wav, sample_rate)
if DEVICE == "cuda":
    torch.cuda.synchronize()
elapsed = time.perf_counter() - t0
seconds = len(wav) / sample_rate
print(
    f"{SAMPLE_PATH.name}: {seconds:.2f} s of audio cleaned in {elapsed:.2f} s "
    f"({seconds / elapsed:.1f}x wall / audio; first GPU call includes warm-up)"
)

background = wav - voice
voice_isolation.save_audio("sample_14_clean.wav", voice, sample_rate, subtype="FLOAT")
voice_isolation.save_audio(
    "sample_14_clean_background.wav", background, sample_rate, subtype="FLOAT"
)
print("wrote sample_14_clean.wav and sample_14_clean_background.wav")

play("Original (noisy)", wav, sample_rate)
play("Cleaned voice", voice, sample_rate)
play("Removed background", background, sample_rate)
print_vram("demo 1 offline")


## 5. Demo 2 — streaming in 20 ms chunks

The realtime model isolates the voice as the audio arrives. The card's streaming snippet pushes **960 samples at a time** (20 ms at 48 kHz) with `lookahead_frames=2`, then `flush()` at the end. The sample stands in for a live source. Times below are this session's wall clock, not a published benchmark.


In [ ]:
CHUNK = 960  # 20 ms at 48 kHz, as in the card's streaming example
assert live.sample_rate == 48_000
assert CHUNK == int(0.020 * live.sample_rate)

wav_48k, sr_48k = voice_isolation.load_audio(str(SAMPLE_PATH))
if sr_48k != live.sample_rate:
    wav_48k = voice_isolation.resample(wav_48k, sr_48k, live.sample_rate)
    sr_48k = live.sample_rate

n_chunks = (len(wav_48k) + CHUNK - 1) // CHUNK
print(
    f"streaming {len(wav_48k) / sr_48k:.2f} s "
    f"({len(wav_48k)} samples) in {n_chunks} chunks of {CHUNK} samples"
)

session = live.stream()
pieces = []
push_times = []
first_audio_s = None

if DEVICE == "cuda":
    torch.cuda.synchronize()
t_loop = time.perf_counter()
for start in range(0, len(wav_48k), CHUNK):
    t_push = time.perf_counter()
    out = session.push(wav_48k[start : start + CHUNK])
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    push_times.append(time.perf_counter() - t_push)
    pieces.append(out)
    if first_audio_s is None and out.numel():
        first_audio_s = time.perf_counter() - t_loop
pieces.append(session.flush())
if DEVICE == "cuda":
    torch.cuda.synchronize()
loop_s = time.perf_counter() - t_loop

streamed = torch.cat(pieces)
print(f"output: {tuple(streamed.shape)} | {streamed.numel() / sr_48k:.2f} s")
print(f"loop wall: {loop_s:.3f} s for {len(wav_48k) / sr_48k:.2f} s of audio")
print(
    f"push: n={len(push_times)} | mean {np.mean(push_times)*1e3:.2f} ms | "
    f"p50 {np.median(push_times)*1e3:.2f} ms | max {np.max(push_times)*1e3:.2f} ms"
)
if first_audio_s is not None:
    print(f"time until first non-empty push: {first_audio_s*1e3:.1f} ms")
else:
    print("time until first non-empty push: (all audio arrived on flush)")
print(f"card algorithmic latency at 20 ms look-ahead: {live.latency_samples() / sr_48k * 1000:.0f} ms")

voice_isolation.save_audio("sample_14_streamed.wav", streamed, sr_48k, subtype="FLOAT")
print("wrote sample_14_streamed.wav")

play("Streamed isolated voice (20 ms chunks)", streamed, sr_48k)
print_vram("demo 2 streaming")


## 6. Optional — isolate your own file

Default is off so **Runtime > Run all** does not block on the Colab file picker. Set `UPLOAD_OWN_AUDIO = True` and re-run this cell to pick a file, or set `MY_RECORDING` to a path already on this machine.

- **Formats:** WAV, FLAC, OGG and MP3 (anything `soundfile` reads). Convert others first, for example `ffmpeg -i input.m4a input.wav`.
- **Sample rate and channels:** any sample rate works. Stereo is mixed down to mono; output is mono.
- **Length:** long files are processed in 8-second chunks that overlap by 2 seconds and are crossfaded.

The cell writes two float32 WAVs beside the source: `*_clean.wav` and `*_clean_background.wav`. Your audio stays on this machine.


In [ ]:
UPLOAD_OWN_AUDIO = False  # set True and re-run this cell to pick a file
MY_RECORDING = ""  # for example "recordings/call.wav"

source = None
if MY_RECORDING:
    source = Path(MY_RECORDING).expanduser()
    if not source.is_file():
        print("MY_RECORDING is set but not found:", source)
        source = None
elif UPLOAD_OWN_AUDIO:
    try:
        from google.colab import files
    except ImportError:
        files = None
    uploaded = files.upload() if files is not None else None
    if not uploaded:
        print("No file uploaded. Put a wav/flac/mp3/ogg in the working directory and set MY_RECORDING.")
    else:
        source = Path(next(iter(uploaded)))
        print("uploaded", source, f"({source.stat().st_size / 1024:.1f} KB)")
else:
    print("Skipped (UPLOAD_OWN_AUDIO = False, MY_RECORDING empty).")

if source is not None:
    original, sr = voice_isolation.load_audio(str(source))
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    cleaned = offline.isolate(original, sr)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    dur = len(original) / sr
    print(f"{source.name}: {dur:.2f} s cleaned in {elapsed:.2f} s")
    removed = original - cleaned
    voice_path = source.with_name(f"{source.stem}_clean.wav")
    background_path = source.with_name(f"{source.stem}_clean_background.wav")
    voice_isolation.save_audio(str(voice_path), cleaned, sr, subtype="FLOAT")
    voice_isolation.save_audio(str(background_path), removed, sr, subtype="FLOAT")
    print(f"saved {voice_path} and {background_path}")
    play("Original", original, sr)
    play("Cleaned voice", cleaned, sr)
    play("Removed background", removed, sr)
    print_vram("demo 3 own file")


## 7. Caveats and cleanup

**What this notebook is for.** A free-T4 (or CPU) demo of Inworld Voice Isolator v1: the card's Python API (`VoiceIsolator("inworld/voice-isolator")` offline, and `mode="streaming", lookahead_frames=2` for 20 ms chunks), on the repo's `sample_14.wav`, with an optional own-file cell.

**What it is not.** A speaker selector. A restoration model (the card: version 1 does not restore missing bandwidth; that is planned for version 2). A VIBE leaderboard run (the eval set is not distributed). The Gradio Offline/Online demo.

**Card limitations (quoted, not retested).** When two people speak at once, the louder voice is kept and one a few dB quieter is removed; voices at similar levels are usually both kept. Smaller GPUs than Ampere/Hopper/Blackwell are not tested yet. The removed background can contain voice leakage.

**API reminders.** Outputs are float32 WAVs at 48 kHz with original levels preserved; voice plus background reconstructs the aligned mono input. `lookahead_frames=2` is 20 ms. Install is `pip install inworld-voice-isolation`. CUDA when available, else CPU.

**T4.** `precision="fp16"` on `sm_75`. No FlashAttention-2. No `[fast]` extra. `HF_TOKEN` is optional (public repo).

**Licence.** Apache-2.0. `sample_14.wav` is the project's bundled example from [inworld-ai/voice-isolator](https://github.com/inworld-ai/voice-isolator).

**Links again:** [model card](https://huggingface.co/inworld/voice-isolator) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/Inworld-Voice-Isolator-48k.md) · [GitHub](https://github.com/inworld-ai/voice-isolator) · [announcement](https://x.com/inworld/status/2123120546908926300)

Free the GPU before you open another notebook:


In [ ]:
print_vram("before cleanup")
for name in (
    "offline",
    "live",
    "session",
    "wav",
    "voice",
    "background",
    "streamed",
    "pieces",
    "wav_48k",
    "original",
    "cleaned",
    "removed",
):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print_vram("after cleanup")
    print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
else:
    print_vram("after cleanup")
    print("CPU run — no VRAM peak. On a T4, print peak VRAM from max_memory_allocated().")
print("Voice Isolator released. Re-run the load cell if you want another isolation.")
